# Disclosure Gap — Deep Dive

**Question:** ESG data mixes company-*reported* figures with provider-*estimated* ones.
The share of estimates is a data-quality signal. Here we go beyond the pillar-level
headline and link the gap to **industry, metric, company, and geography**.

Source: `long_clean.parquet` (1.8M observations, one per company × metric × year),
joined to `company_meta.parquet` for industry. `disclosure ∈ {REPORTED, ESTIMATED}`.


In [1]:
import pandas as pd, numpy as np
import plotly.express as px

lc = pd.read_parquet("../outputs/long_clean.parquet")[
    ["perm_id","company_name","disclosure","metric_name","pillar","category","headquarter_country"]]
meta = pd.read_parquet("../outputs/company_meta.parquet")[["perm_id","industry"]].drop_duplicates()

df = lc.merge(meta, on="perm_id", how="left")
df["est"] = (df["disclosure"] == "ESTIMATED").astype(int)   # 1 = estimated, 0 = reported

print(f"{len(df):,} observations | {df.perm_id.nunique():,} companies | "
      f"{df.industry.nunique()} industries | overall estimated share: {df.est.mean():.1%}")
df.head(3)

1,816,988 observations | 65,069 companies | 78 industries | overall estimated share: 42.0%


,perm_id,company_name,disclosure,metric_name,pillar,category,headquarter_country,industry,est
0,4295892049,Formosa Sumco Technology Corp,REPORTED,ANALYTIC_AUDIT_COMM_EXPERTISE,G,Governance Opportunity,"Taiwan, China",Semiconductors,0
1,4295857926,Maxiparts Ltd,REPORTED,ANALYTIC_VOTING_RIGHTS,G,Governance Opportunity,Australia,Software & IT Services,0
2,4297024126,MaxCyte Inc,REPORTED,ANALYTIC_VOTING_RIGHTS,G,Governance Opportunity,United States,Biotechnology & Pharmaceuticals,0


## 0. Baseline — the pillar-level gap

The headline finding, for context: the Environmental pillar is overwhelmingly
*estimated*, while Social and Governance are mostly *reported*.


In [2]:
pillar = (df.groupby("pillar")["est"].mean().mul(100).round(1)
            .rename("estimated_%").reset_index())
fig = px.bar(pillar, x="pillar", y="estimated_%", color="pillar",
             text="estimated_%", title="Estimated share by pillar",
             color_discrete_map={"E":"#2e8b57","S":"#4682b4","G":"#9370db"})
fig.update_yaxes(title="% estimated", range=[0,100]); fig.show()
pillar

,pillar,estimated_%
0,E,71.4
1,G,11.6
2,S,11.1


**Read:** ~71% of Environmental observations are modelled by the provider vs ~11%
for S and G. Companies *state* their policies (S/G) but their *quantitative*
environmental figures (emissions, water, waste) are mostly estimated. Everything
below explains *where* that gap concentrates.


## 1. By industry — who self-reports, who relies on estimates

Restricting to industries with ≥3,000 observations for stable rates.


In [3]:
ind = df.groupby("industry")["est"].agg(estimated_share="mean", n="count")
ind = ind[ind["n"] >= 3000].sort_values("estimated_share")
ind["estimated_%"] = (ind["estimated_share"]*100).round(1)

top_bottom = pd.concat([ind.head(10), ind.tail(10)])
fig = px.bar(top_bottom.reset_index(), x="estimated_%", y="industry", orientation="h",
             title="Most self-reported (top) vs most estimated (bottom) industries",
             labels={"estimated_%":"% estimated"})
fig.update_layout(yaxis={"categoryorder":"total descending"}, height=650); fig.show()

**Read:** a ~2× spread — transport & utilities (Airlines ~27%, Rail, Electric
Utilities) self-report the most, likely because emissions reporting is core to
their regulation and operations. Asset-light / consumer sectors (Apparel ~52%,
Advertising, Asset Management) rely most on estimates.


## 2. Metric × industry — where a *specific* metric is actually reported

Pillar/industry averages hide metric-level truth. Pick a material metric and see
which industries report it vs have it estimated. Change `KEY` to explore others.


In [4]:
KEY = "CO2DIRECTSCOPE1"   # Scope 1 direct emissions
sub = df[df.metric_name == KEY]
mi = sub.groupby("industry")["est"].agg(estimated_share="mean", n="count")
mi = mi[mi["n"] >= 50].sort_values("estimated_share")
mi["estimated_%"] = (mi["estimated_share"]*100).round(1)

fig = px.bar(mi.reset_index(), x="estimated_%", y="industry", orientation="h",
             title=f"{KEY}: estimated share by industry",
             labels={"estimated_%":"% estimated"})
fig.update_layout(yaxis={"categoryorder":"total descending"}, height=700); fig.show()

**Read:** even for the *same* metric, disclosure varies enormously — Airlines/Rail/
Autos actually measure Scope 1 (~64–70% estimated), while Biofuels, Mortgage
Finance and Advertising almost never report it (>92% estimated). A single
industry-agnostic "carbon estimate" therefore hides very different data quality.


In [5]:
# Full metric x pillar heatmap: estimated-share for the most-common metrics
top_metrics = df.metric_name.value_counts().head(30).index
piv = (df[df.metric_name.isin(top_metrics)]
       .groupby(["metric_name","pillar"])["est"].mean().mul(100).round(0)
       .reset_index())
fig = px.bar(piv, x="est", y="metric_name", color="pillar", orientation="h",
             title="Estimated share for the 30 most-common metrics",
             labels={"est":"% estimated"},
             color_discrete_map={"E":"#2e8b57","S":"#4682b4","G":"#9370db"})
fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=750); fig.show()

## 3. By company — drill into a single firm's disclosure profile

Two views: (a) rank companies by how much of their data is estimated, and
(b) inspect one company's reported-vs-estimated split.


In [6]:
# (a) companies ranked by estimated share (min 20 observations each)
co = df.groupby(["perm_id","company_name","industry"])["est"].agg(
        estimated_share="mean", n="count").reset_index()
co = co[co["n"] >= 20]
co["estimated_%"] = (co["estimated_share"]*100).round(1)
print("Most estimate-reliant companies:")
display(co.sort_values("estimated_share", ascending=False).head(10)[
        ["company_name","industry","estimated_%","n"]])
print("Most self-reported companies:")
display(co.sort_values("estimated_share").head(10)[
        ["company_name","industry","estimated_%","n"]])

Most estimate-reliant companies:


,company_name,industry,estimated_%,n
5189,Europlasma SA,Electrical & Electronic Equipment,85.0,20
2758,Bestsun Energy Co Ltd,Engineering & Construction Services,85.0,20
11139,Toa Oil Co Ltd,Oil & Gas Refining & Marketing,85.0,20
46648,Prio Forte SA,Oil & Gas Exploration & Production,85.0,20
37091,Jiangsu Linyang Energy Co Ltd,Electrical & Electronic Equipment,85.0,20
28361,Ukrnafta PAT,Software & IT Services,85.0,20
13247,EO Technics Co Ltd,Semiconductors,85.0,20
27241,Titas Gas Transmission and Distribution Compan...,Gas Utilities & Distributors,81.0,21
2951,China Nonferrous Metal Industry's Foreign Engi...,Metals & Mining,81.0,21
2787,Shanghai Datun Energy Resources Co Ltd,Coal Operations,81.0,21


Most self-reported companies:


,company_name,industry,estimated_%,n
61583,Fcc Servicios Medio Ambiente Holding SA,Solar Technology & Project Developers,0.0,31
43237,Zahid Group Holding Mena Ltd,Toys & Sporting Goods,0.0,37
57164,Abdulkadir Al Muhaidib and Sons LLC,Industrial Machinery & Goods,0.0,37
44708,Agrosuper SA,Electrical & Electronic Equipment,0.0,50
39626,National Petrochemical Industrial Co,Chemicals,0.0,47
32456,Piraeus Group Finance PLC,Consumer Finance,0.0,24
47117,Fozan Holding Co,"Apparel, Accessories & Footwear",0.0,37
52327,Imago BioSciences Inc,Biotechnology & Pharmaceuticals,0.0,32
52286,Iochpe-Maxion Austria GmbH,Leisure Facilities,0.0,30
51058,Modern Industrial Investment Holding Group,Hardware,0.0,37


In [7]:
# (b) one company's breakdown — set NAME to any company_name in the data
NAME = co.sort_values("n", ascending=False).iloc[0]["company_name"]
one = df[df.company_name == NAME]
brk = one.groupby("disclosure")["metric_name"].count().rename("metrics")
print(f"{NAME} — {one.industry.iloc[0]}")
print(f"reported vs estimated: {brk.to_dict()}")
print("\nEstimated metrics for this company:")
print(sorted(one[one.disclosure=='ESTIMATED'].metric_name.unique())[:20])

Repsol SA — Oil & Gas Exploration & Production
reported vs estimated: {'CALCULATED': 4, 'ESTIMATED': 8, 'REPORTED': 81}

Estimated metrics for this company:
['AIRPOLLUTANTS_DIRECT', 'AIRPOLLUTANTS_INDIRECT', 'BRIBERY_AND_CORRUPTION_PAI_INSUFFICIENT_ACTIONS', 'HUMAN_RIGHTS_VIOLATION_PAI', 'NATURAL_RESOURCE_USE_DIRECT', 'NOXEMISSIONS', 'SOXEMISSIONS', 'WATER_USE_PAI_M10']


## 4. By geography — disclosure culture by headquarter country

Countries with ≥5,000 observations.


In [8]:
geo = df.groupby("headquarter_country")["est"].agg(estimated_share="mean", n="count")
geo = geo[geo["n"] >= 5000].sort_values("estimated_share")
geo["estimated_%"] = (geo["estimated_share"]*100).round(1)
tb = pd.concat([geo.head(10), geo.tail(10)])
fig = px.bar(tb.reset_index(), x="estimated_%", y="headquarter_country", orientation="h",
             title="Most self-reported (top) vs most estimated (bottom) countries",
             labels={"estimated_%":"% estimated","headquarter_country":"country"})
fig.update_layout(yaxis={"categoryorder":"total descending"}, height=650); fig.show()

**Read:** a strong geographic gradient — firms HQ'd in Ireland, Netherlands,
Switzerland, Saudi Arabia disclose most directly (~21–24% estimated), while
Bangladesh (81%), Vietnam, Pakistan rely heavily on estimates. Mandatory-reporting
regimes (EU) and large-cap concentration likely drive the difference.


## 5. Structural pattern — *what kind* of metric never gets reported

The clearest signal: **policy/flag** metrics are ~always reported (companies assert
they have a policy), while **quantitative PAI / pollutant** metrics are ~always
estimated (companies don't measure them, so the provider models them).


In [9]:
mm = df.groupby("metric_name")["est"].agg(estimated_share="mean", n="count")
mm = mm[mm["n"] >= 5000]
mm["estimated_%"] = (mm["estimated_share"]*100).round(1)
print("Always ESTIMATED (companies don't measure these):")
display(mm.sort_values("estimated_share", ascending=False).head(12)[["estimated_%","n"]])
print("Always REPORTED (companies self-assert these):")
display(mm.sort_values("estimated_share").head(12)[["estimated_%","n"]])

Always ESTIMATED (companies don't measure these):


,estimated_%,n
metric_name,,
AIRPOLLUTANTS_DIRECT,100.0,13579
BRIBERY_AND_CORRUPTION_PAI_INSUFFICIENT_ACTIONS,100.0,50526
HUMAN_RIGHTS_VIOLATION_PAI,100.0,50526
WATER_USE_PAI_M10,100.0,50526
AIRPOLLUTANTS_INDIRECT,100.0,45450
VOCEMISSIONS,98.6,55010
PARTICULATE_MATTER_EMISSIONS,98.0,55552
NOXEMISSIONS,97.1,55960
SOXEMISSIONS,97.0,55886


Always REPORTED (companies self-assert these):


,estimated_%,n
metric_name,,
HUMAN_RIGHTS_POLICY_DUEDILIGENCE,0.0,9746
POLICY_FREEDOMOF_ASSOCIATION,0.0,10446
POLICY_FORCED_LABOR,0.0,17412
POLICY_EMISSIONS,0.0,25318
POLICY_DATA_PRIVACY,0.0,18892
POLICY_CHILD_LABOR,0.0,18213
POLICY_BRIBERYAND_CORRUPTION,0.0,23351
POLICY_HUMAN_RIGHTS,0.0,31673
POLICY_BOARD_DIVERSITY,0.0,14042


## Takeaways

1. **The gap is an *Environmental*, *quantitative* problem** — E is 71% estimated;
   S/G ~11%. Policy flags are reported, measured quantities are not.
2. **Industry matters ~2×** — regulated emitters (airlines, utilities) self-report;
   asset-light consumer sectors don't.
3. **Even one metric splits by industry** — CO₂ Scope 1 is measured by transport,
   modelled for finance/ads. Treating any ESG metric as uniformly reliable is wrong.
4. **Geography adds a strong prior** — EU/developed HQ ⇒ far more direct disclosure.
5. **Actionable for the app:** add industry / metric / company / country filters to
   the Disclosure page, and precompute `disclosure_by_industry` and
   `disclosure_by_company` tables so the SQL console can query them.
